# Understanding Maji Ndogo's agriculture
© ExploreAI Academy

## Introduction
Hey there, I'm glad you're on board for the Maji Ndogo project AGAIN! Let me walk you through what we're up against and how we'll tackle it.

As you know, we're in an ambitious project aimed at automating farming in Maji Ndogo, a place with diverse and challenging agricultural landscapes. Before we dive into the 'how' of farming, we need to figure out the 'where' and 'what'. It's not just about deploying technology; it's about making informed decisions on where to plant specific crops, considering factors like rainfall, soil type, climate, and many others.

Our analysis is the groundwork for this entire automation project. We have an array of variables like soil fertility, climate conditions, and geographical data. By understanding these elements, we can recommend the best locations for different crops. It's a bit like solving a complex puzzle – each piece of data is crucial to seeing the bigger picture.

We'll start by importing our dataset into a DataFrame. It is currently in an SQLite database, and split into tables. Unlike Power BI or SQL, data analysis in Python happens in a single table. So we will have to brush off those dusty SQL skills to get the data imported. Expect a bit of a mess in the data – it's part of the challenge. We need to clean it up and maybe reshape it to make sense of it. It's like sorting out the tools and materials we need and getting rid of what we don't.

Here's where the real fun begins. We'll dive deep into the data, looking for patterns, and correlations. Each clue in the data leads us closer to understanding the best farming practices for Maji Ndogo. I'll be relying on your skills and insights. We'll be working through these steps together, discussing our findings and strategies.

Let's gear up and get ready to make a real difference in Maji Ndogo. Ready to get started? Let's dive into our data and see what stories it has to tell us.

Sanaa.


## Data dictionary
**1. Geographic features**

- **Field_ID:** A unique identifier for each field (BigInt).
 
- **Elevation:** The elevation of the field above sea level in metres (Float).

- **Latitude:** Geographical latitude of the field in degrees (Float).

- **Longitude:** Geographical longitude of the field in degrees (Float).

- **Location:** Province the field is in (Text).

- **Slope:** The slope of the land in the field (Float).

**2. Weather features**

- **Field_ID:** Corresponding field identifier (BigInt).

- **Rainfall:** Amount of rainfall in the area in mm (Float).

- **Min_temperature_C:** Average minimum temperature recorded in Celsius (Float).

- **Max_temperature_C:** Average maximum temperature recorded in Celsius (Float).

- **Ave_temps:** Average temperature in Celcius (Float).

**3. Soil and crop features**

- **Field_ID:** Corresponding field identifier (BigInt).

- **Soil_fertility:** A measure of soil fertility where 0 is infertile soil, and 1 is very fertile soil (Float).

- **Soil_type:** Type of soil present in the field (Text).

- **pH:** pH level of the soil, which is a measure of how acidic/basic the soil is (Float).

**4. Farm management features**

- **Field_ID:** Corresponding field identifier (BigInt).

- **Pollution_level:** Level of pollution in the area where 0 is unpolluted and 1 is very polluted (Float).

- **Plot_size:** Size of the plot in the field (Ha) (Float).

- **Chosen_crop:** Type of crop chosen for cultivation (Text).

- **Annual_yield:** Annual yield from the field (Float). This is the total output of the field. The field size and type of crop will affect the Annual Yield

- **Standard_yield:** Standardised yield expected from the field, normalised per crop (Float). This is independent of field size, or crop type. Multiplying this number by the field size, and average crop yield will give the Annual_Yield.

In [60]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

In [61]:
# Printing out the names of tables in the database
dataset = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", engine)

dataset

,name
0,geographic_features
1,weather_features
2,soil_and_crop_features
3,farm_management_features


In [62]:
# Writing a SQL query to join all the tables together
# Since we have no need for the Field_ID at this time, it will be omitted
sql_query = """
SELECT
    geo.Elevation,
    geo.Latitude,
    geo.Longitude,
    geo.Location,
    geo.Slope,

    w.Rainfall,
    w.Min_temperature_C,
    w.Max_temperature_C,
    w.Ave_temps,

    sc.Soil_fertility,
    sc.Soil_type,
    sc.pH,

    fm.Pollution_level,
    fm.Plot_size,
    fm.Crop_type,
    fm.Annual_yield,
    fm.Standard_yield

FROM
    geographic_features as geo
JOIN
    weather_features as w
    ON
        geo.Field_ID = w.Field_ID
JOIN
    soil_and_crop_features as sc
    ON
        geo.Field_ID = sc.Field_ID
JOIN
    farm_management_features as fm
    ON
        geo.Field_ID = fm.Field_ID
"""

MD_agric_df = pd.read_sql_query(sql_query, engine)
MD_agric_df

,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Crop_type,Annual_yield,Standard_yield
0,786.05580,-7.389911,-7.556202,Rural_Akatsi,14.795113,1125.2,-3.1,33.1,15.00,0.62,Sandy,6.169393,8.526684e-02,1.3,0.751354,cassava,0.577964
1,674.33410,-7.736849,-1.051539,Rural_Sokoto,11.374611,1450.7,-3.9,30.6,13.35,0.64,Volcanic,5.676648,3.996838e-01,2.2,1.069865,cassava,0.486302
2,826.53390,-9.926616,0.115156,Rural_Sokoto,11.339692,2208.9,-1.8,28.4,13.30,0.69,Volcanic,5.331993,3.580286e-01,3.4,2.208801,tea,0.649647
3,574.94617,-2.420131,-6.592215,Rural_Kilimani,7.109855,328.8,-5.8,32.2,13.20,0.54,Loamy,5.328150,2.866871e-01,2.4,1.277635,cassava,0.532348
4,886.35300,-3.055434,-7.952609,Rural_Kilimani,55.007656,785.2,-2.5,31.0,14.25,0.72,Sandy,5.721234,4.319027e-02,1.5,0.832614,wheat,0.555076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5649,681.36145,-7.358371,-6.254369,Rural_Akatsi,16.213196,885.7,-4.3,33.4,14.55,0.61,Sandy,5.741063,3.286828e-01,1.1,0.609930,potato,0.554482
5650,667.02120,-3.154559,-4.475046,Rural_Kilimani,2.397553,501.1,-4.8,32.1,13.65,0.54,Sandy,5.445833,1.602583e-01,8.7,3.812289,maize,0.438194
5651,670.77900,-14.472861,-6.110221,Rural_Hawassa,7.636470,1586.6,-3.8,33.4,14.80,0.64,Volcanic,5.385873,8.221326e-09,2.1,1.681629,tea,0.800776
5652,429.48840,-14.653089,-6.984116,Rural_Hawassa,13.944720,1272.2,-6.2,34.6,14.20,0.63,Silt,5.562508,6.917245e-10,1.3,0.659874,cassava,0.507595


In [64]:
# Minimum Elevation value
print(MD_agric_df['Elevation'].min())

-878.8608


In [66]:
# Checking the data information for Annual yield and crop type column
MD_agric_df[['Annual_yield', 'Crop_type']].info()

<class 'pandas.DataFrame'>
RangeIndex: 5654 entries, 0 to 5653
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Annual_yield  5654 non-null   str    
 1   Crop_type     5654 non-null   float64
dtypes: float64(1), str(1)
memory usage: 88.5 KB


In [71]:
# Listing the unique values in Annual yield
print(MD_agric_df['Annual_yield'].unique())

<StringArray>
[ 'cassava',      'tea',    'wheat',   'potato',   'banana',   'coffee',
     'rice',    'maize',   'wheat ',     'tea ', 'cassaval',   'wheatn',
 'cassava ',     'teaa']
Length: 14, dtype: str


## Data cleanup
I noticed some errors in the data. Here's what I picked up:

1. There are some swapped column names. 

2. Some of the crop types contain spelling errors.

3.  The `Elevation` column contains some negative values, which are not plausible, so change these to positive values.

Use your Pandas skills to clean up the data.

In [72]:
# Removing the negatives values in the elevation column
MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()

# Swapping the column name for Annual_yield with Crop_type
MD_agric_df.rename(
    columns={
        'Annual_yield': 'Crop_type',
        'Crop_type': 'Annual_yield'
    }, inplace=True
)

# Correcting the misspelled names in Crop_type column
correct_name = {
    'cassaval': 'cassava',
    'cassava ': 'cassava',
    'tea ': 'tea',
    'teaa': 'tea',
    'wheat ': 'wheat',
    'wheatn': 'wheat'
}
MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].replace(correct_name)
MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].str.strip().str.lower()

MD_agric_df.head()

,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
0,786.05580,-7.389911,-7.556202,Rural_Akatsi,14.795113,1125.2,-3.1,33.1,15.00,0.62,Sandy,6.169393,0.085267,1.3,0.751354,cassava,0.577964
1,674.33410,-7.736849,-1.051539,Rural_Sokoto,11.374611,1450.7,-3.9,30.6,13.35,0.64,Volcanic,5.676648,0.399684,2.2,1.069865,cassava,0.486302
2,826.53390,-9.926616,0.115156,Rural_Sokoto,11.339692,2208.9,-1.8,28.4,13.30,0.69,Volcanic,5.331993,0.358029,3.4,2.208801,tea,0.649647
3,574.94617,-2.420131,-6.592215,Rural_Kilimani,7.109855,328.8,-5.8,32.2,13.20,0.54,Loamy,5.328150,0.286687,2.4,1.277635,cassava,0.532348
4,886.35300,-3.055434,-7.952609,Rural_Kilimani,55.007656,785.2,-2.5,31.0,14.25,0.72,Sandy,5.721234,0.043190,1.5,0.832614,wheat,0.555076


### Data Checkup

In [79]:
print(f"Crop type unique values:")
print(list(MD_agric_df['Crop_type'].unique()))

print(f"\nMinimum Elevation value = {MD_agric_df['Elevation'].min()}")

print(f"\nData type for Annual yield and Crop type:")
print(f"  1. Annual yield data type: {MD_agric_df['Annual_yield'].dtype}")
print(f"  2. Crop type data type: {MD_agric_df['Crop_type'].dtype}")

Crop type unique values:
['cassava', 'tea', 'wheat', 'potato', 'banana', 'coffee', 'rice', 'maize']

Minimum Elevation value = 35.910797

Data type for Annual yield and Crop type:
  1. Annual yield data type: float64
  2. Crop type data type: str


### Challenge 1: Uncovering crop preferences
Now that we have our data ready, let's delve into understanding where different crops are grown in Maji Ndogo. Our initial step is to focus on tea, a key crop in Maji Ndogo. We need to determine the optimal conditions for its growth. By analyzing data related to elevation, rainfall, and soil type specifically for tea plantations, we'll start to paint a picture of where our farming systems could thrive.

**Task:**
Create a function that includes only tea fields and returns a tuple with the mean `Rainfall` and the mean `Elevation`. The function should input the full DataFrame, a string value for the crop type to filter by, and output a tuple with rainfall and elevation.


In [105]:
def explore_crop_distribution(df, crop):
    """
    This function calculates the mean elevation and rainfall
    for tea crop.

    input:
     - df: dataframe conatining the data
     - crop (str): The selected crop to be evaluated

     output:
     - tuple: containing the mean values of elevation and 
       rainfall
    """
    filtered_crop = df[(df['Crop_type'] == crop)]
    mean_values = tuple(filtered_crop[['Rainfall', 'Elevation']].mean())
    return mean_values

Input: `explore_crop_distribution(MD_agric_df, 'tea')`

Expected output: `(1534.5079956188388, 775.208667535597)`

In [122]:
explore_crop_distribution(MD_agric_df, 'tea')

(1534.5079956188388, 775.208667535597)

input: `explore_crop_distribution(MD_agric_df, "wheat")`

Expected output: `(1010.2859910581222, 595.8384148002981)`

In [100]:
explore_crop_distribution(MD_agric_df, "wheat")

(1010.2859910581222, 595.8384148002981)

### Challenge 2: Finding fertile grounds
With insights into tea cultivation, let's broaden our horizons. Fertile soil is the bedrock of successful farming. By grouping our data by location and soil type, we'll pinpoint where the most fertile soils in Maji Ndogo are. These fertile zones could be prime candidates for diverse crop cultivation, maximising our yield.

We’ll group our data by soil type to see where the most fertile grounds are. This information will be vital for deciding where to deploy our farming technology.

**Task:** Create a function that groups the data by `Soil_type`, and returns the `Soil_fertility`.


In [98]:
def analyse_soil_fertility(df):
    """
    List the most fertile soil in Maji Ndogo
    """
    soil_fertility = df.groupby(['Soil_type'])['Soil_fertility'].mean()
    return soil_fertility

Input: `analyse_soil_fertility(MD_agric_df)`

Expected output:
```python Soil_Type
Loamy       0.585868
Peaty       0.604882
Rocky       0.582368
Sandy       0.595669
Silt        0.652654
Volcanic    0.648894
Name: Soil_Fertility, dtype: float64
```

In [99]:
analyse_soil_fertility(MD_agric_df)

Soil_type
Loamy       0.585868
Peaty       0.604882
Rocky       0.582368
Sandy       0.595669
Silt        0.652654
Volcanic    0.648894
Name: Soil_fertility, dtype: float64

### Challenge 3: Climate and geography analysis
Now, let's delve into how climate and geography influence farming. By understanding the relationship between factors like elevation, temperature, and rainfall with crop yields, we can identify the most suitable areas for different crops. This analysis is key to ensuring our automated systems are deployed in locations that will maximise their effectiveness.

**Task:** Create a function that takes in a DataFrame and the column name, and groups the data by that column, and aggregates the data by the means of `Elevation`, `Min_temperature_C`, `Max_temperature_C`, and `Rainfall`, and outputs a DataFrame. Please ensure that the order of the columns matches the output.


In [123]:
def climate_geography_influence(df, column):
    """
    calculates the mean values for Elevation, Minimum temperature, Maximum temperature, and Rainfall for each crop
    """
    cols = [column, 'Elevation', 'Min_temperature_C', 'Max_temperature_C', 'Rainfall']
    return df[cols].groupby(column).mean()

input: `climate_geography_influence(MD_agric_df, 'Crop_type')`

Expected output:

```sql
Crop_type 	Elevation	Min_temperature_C	Max_temperature_C	Rainfall			
banana		487.973572	-5.354344		31.988152	    1659.905687
cassava		682.903008	-3.992113		30.902381	    1210.543006
coffee		647.047734	-4.028007		30.855189	    1527.265074
maize		680.596982	-4.497995		30.576692	    681.010276
potato		696.313917	-4.375334		30.300608	    660.289064
rice		352.858053	-6.610566		32.727170	    1632.382642
tea		775.208668	-2.862651		29.950383	    1534.507996
wheat		595.838415	-4.968107		30.973845	    1010.285991
```

In [124]:
climate_geography_influence(MD_agric_df, 'Crop_type')

,Elevation,Min_temperature_C,Max_temperature_C,Rainfall
Crop_type,,,,
banana,487.973572,-5.354344,31.988152,1659.905687
cassava,682.903008,-3.992113,30.902381,1210.543006
coffee,647.047734,-4.028007,30.855189,1527.265074
maize,680.596982,-4.497995,30.576692,681.010276
potato,696.313917,-4.375334,30.300608,660.289064
rice,352.858053,-6.610566,32.727170,1632.382642
tea,775.208668,-2.862651,29.950383,1534.507996
wheat,595.838415,-4.968107,30.973845,1010.285991


### Challenge 4: Advanced sorting techniques
Quite often it is better to improve the things you're good at than improving the things you're bad at. So the question is, which crop is the top performer in Maji Ndogo, and under what conditions does it perform well? 

To answer this, we need to:
1. Filter all the fields with an above-average `Standard_yield` (do you have flashbacks to SQL subqueries right now?).
2. Then group by <?> crop type, using `count()`.
3. Sort the values to get the top crop type on top.
4. Retrieve the name of the top index. See the hint below on how to do this. 

**Task:** Create a function that takes a DataFrame as input, filters, groups and sorts, and outputs a string value of a crop type.

In [137]:
def find_ideal_fields(df):
    """
    Compares the mean standard yields with the standard yields of each crop 
    and returns the crop with the highest standard yield
    """
    mean_sy = df['Standard_yield'].mean()
    filtered_sy = df[df['Standard_yield'] > mean_sy]
    top_crop = filtered_sy.groupby('Crop_type').count().sort_values(by='Standard_yield', ascending=False).index[0]

    return top_crop

input: `find_ideal_fields(MD_agric_df)`

Expected output: `tea`

In [138]:
find_ideal_fields(MD_agric_df)

'tea'

### Challenge 5: Advanced filtering techniques
Now we know that <?> is our most successful crop, we can look at what makes it successful.

Create a function that takes a DataFrame as input, and the type of crop, and filters the DataFrame using the following conditions:
1. Filter by crop type.

2. Select only rows that have above average `Standard_yield`.

3. Select only rows that have `Ave_temps` >= 12 but =< 15.

4. Have a `Pollution_level` lower than 0.0001.

In [145]:
def find_good_conditions(df, crop_type):
    
    mean_standard_yield = df['Standard_yield'].mean()

    crop_filter = df['Crop_type'] == crop_type
    high_yield = df['Standard_yield'] > mean_standard_yield
    temp = (df['Ave_temps'] >= 12) & (df['Ave_temps'] <= 15)
    pollution_level = df['Pollution_level'] < 0.0001

    conditions = df[crop_filter & high_yield & temp & pollution_level]

    return conditions

Input: `find_good_conditions(MD_agric_df, "tea").shape`

Expected output: `(14, 17)`

In [148]:
find_good_conditions(MD_agric_df, "tea").shape

(14, 17)

In [147]:
find_good_conditions(MD_agric_df, "tea")

,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
197,688.63477,-14.585503,-5.948055,Rural_Hawassa,2.506081,1569.9,-3.6,31.6,14.00,0.62,Volcanic,4.117729,1.007224e-08,10.4,7.484284,tea,0.719643
852,739.42730,-14.405275,-6.267883,Rural_Hawassa,7.132496,1687.9,-3.1,31.1,14.00,0.64,Volcanic,4.538485,7.108529e-09,2.1,1.664479,tea,0.792609
869,691.24520,-14.400770,-5.966074,Rural_Hawassa,11.563782,1640.5,-3.5,32.7,14.60,0.65,Volcanic,4.123869,1.691890e-08,3.0,2.219570,tea,0.739857
943,713.74615,-14.549458,-6.051661,Rural_Hawassa,14.432365,1604.0,-3.4,27.7,12.15,0.66,Volcanic,5.147911,7.575035e-09,0.7,0.554950,tea,0.792785
1293,648.13605,-14.297139,-6.146258,Rural_Hawassa,9.684363,1648.5,-3.8,29.5,12.85,0.65,Volcanic,5.154659,1.387864e-08,2.5,2.095446,tea,0.838179
1376,660.65173,-14.423298,-6.011120,Rural_Hawassa,7.688627,1603.4,-3.8,28.8,12.50,0.64,Volcanic,4.330087,1.364089e-08,3.4,2.648997,tea,0.779117
2010,527.61390,-14.161968,-6.731858,Rural_Hawassa,0.977717,1560.2,-5.1,33.3,14.10,0.61,Peaty,5.092398,1.150044e-08,9.3,6.679613,tea,0.718238
2278,667.37920,-14.382747,-6.632757,Rural_Hawassa,2.563133,1625.9,-3.8,31.4,13.80,0.62,Volcanic,4.844913,4.868797e-09,14.3,11.751984,tea,0.821817
3445,678.59955,-14.648583,-6.083194,Rural_Hawassa,16.919823,1531.3,-3.8,28.4,12.30,0.66,Volcanic,4.828436,4.663686e-09,2.0,1.580439,tea,0.790220
3568,684.76980,-14.261093,-6.267883,Rural_Hawassa,21.928755,1689.2,-3.5,29.2,12.85,0.69,Volcanic,4.815771,1.248680e-08,0.6,0.499929,tea,0.833216
